In [ ]:
# 실행 환경 확인
from pathlib import Path
import importlib.util
import sys

required_packages = ["numpy", "scipy", "pandas", "matplotlib", "plotly", "numba", "cupy"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f"Missing packages: {missing_packages}. Install requirements.txt first.")

import numpy as np
from dataclasses import dataclass
import matplotlib
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

matplotlib.rcParams["font.family"] = "Malgun Gothic" if sys.platform.startswith("win") else "DejaVu Sans"
matplotlib.rcParams["axes.unicode_minus"] = False

print("실행 폴더:", ROOT)


In [ ]:
# 목표 전기장 함수 정의
EPS0 = 8.8541878128e-12


@dataclass
class BoxConfig:
    Lx: float = 1.30
    Ly: float = 0.85
    Lz: float = 1.10
    Nx: int = 65
    Ny: int = 51
    Nz: int = 57
    E0: float = 1.0
    seed: int = 12
    poly_scale: float = 0.18
    fourier_scale: float = 0.12
    n_fourier_terms: int = 5
    use_polynomial: bool = True
    use_fourier: bool = True
    dtype: type = np.float64


def make_grid(cfg: BoxConfig):
    x = np.linspace(0.0, cfg.Lx, cfg.Nx, dtype=cfg.dtype)
    y = np.linspace(0.0, cfg.Ly, cfg.Ny, dtype=cfg.dtype)
    z = np.linspace(0.0, cfg.Lz, cfg.Nz, dtype=cfg.dtype)

    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    Xc = X - 0.5 * cfg.Lx
    Yc = Y - 0.5 * cfg.Ly
    Zc = Z - 0.5 * cfg.Lz

    a = max(cfg.Lx, cfg.Ly, cfg.Lz)
    u = Xc / a
    v = Yc / a
    w = Zc / a
    wmax = 0.5 * cfg.Lz / a

    return x, y, z, X, Y, Z, Xc, Yc, Zc, u, v, w, a, wmax


def sample_poly_coefficients(seed=0, scale=0.20):
    rng = np.random.default_rng(seed)

    names = [
        "u",
        "v",
        "w",
        "uv",
        "uw",
        "vw",
        "u2_minus_v2",
        "2w2_minus_u2_minus_v2",
        "u_4w2_u2_v2",
        "v_4w2_u2_v2",
        "w_u2_minus_v2",
        "uvw",
        "u_u2_minus_3v2",
        "v_3u2_minus_v2",
        "w_2w2_minus_3u2_minus_3v2",
    ]

    coeffs = dict(zip(names, rng.normal(0.0, scale, size=len(names))))
    coeffs["u"] += 0.55
    coeffs["v"] -= 0.30
    coeffs["w"] += 0.20
    return coeffs


def add_polynomial_harmonics(phi, Ex, Ey, Ez, u, v, w, coeffs, E0, a):
    S = E0 * a

    c = coeffs.get("u", 0.0)
    phi += S * c * u
    Ex += -E0 * c

    c = coeffs.get("v", 0.0)
    phi += S * c * v
    Ey += -E0 * c

    c = coeffs.get("w", 0.0)
    phi += S * c * w
    Ez += -E0 * c

    c = coeffs.get("uv", 0.0)
    phi += S * c * (u * v)
    Ex += -E0 * c * v
    Ey += -E0 * c * u

    c = coeffs.get("uw", 0.0)
    phi += S * c * (u * w)
    Ex += -E0 * c * w
    Ez += -E0 * c * u

    c = coeffs.get("vw", 0.0)
    phi += S * c * (v * w)
    Ey += -E0 * c * w
    Ez += -E0 * c * v

    c = coeffs.get("u2_minus_v2", 0.0)
    phi += S * c * (u * u - v * v)
    Ex += -E0 * c * (2.0 * u)
    Ey += +E0 * c * (2.0 * v)

    c = coeffs.get("2w2_minus_u2_minus_v2", 0.0)
    phi += S * c * (2.0 * w * w - u * u - v * v)
    Ex += +E0 * c * (2.0 * u)
    Ey += +E0 * c * (2.0 * v)
    Ez += -E0 * c * (4.0 * w)

    c = coeffs.get("u_4w2_u2_v2", 0.0)
    phi += S * c * (u * (4.0 * w * w - u * u - v * v))
    Ex += -E0 * c * (4.0 * w * w - 3.0 * u * u - v * v)
    Ey += +E0 * c * (2.0 * u * v)
    Ez += -E0 * c * (8.0 * u * w)

    c = coeffs.get("v_4w2_u2_v2", 0.0)
    phi += S * c * (v * (4.0 * w * w - u * u - v * v))
    Ex += +E0 * c * (2.0 * u * v)
    Ey += -E0 * c * (4.0 * w * w - u * u - 3.0 * v * v)
    Ez += -E0 * c * (8.0 * v * w)

    c = coeffs.get("w_u2_minus_v2", 0.0)
    phi += S * c * (w * (u * u - v * v))
    Ex += -E0 * c * (2.0 * u * w)
    Ey += +E0 * c * (2.0 * v * w)
    Ez += -E0 * c * (u * u - v * v)

    c = coeffs.get("uvw", 0.0)
    phi += S * c * (u * v * w)
    Ex += -E0 * c * (v * w)
    Ey += -E0 * c * (u * w)
    Ez += -E0 * c * (u * v)

    c = coeffs.get("u_u2_minus_3v2", 0.0)
    phi += S * c * (u * (u * u - 3.0 * v * v))
    Ex += -E0 * c * (3.0 * u * u - 3.0 * v * v)
    Ey += +E0 * c * (6.0 * u * v)

    c = coeffs.get("v_3u2_minus_v2", 0.0)
    phi += S * c * (v * (3.0 * u * u - v * v))
    Ex += -E0 * c * (6.0 * u * v)
    Ey += -E0 * c * (3.0 * u * u - 3.0 * v * v)

    c = coeffs.get("w_2w2_minus_3u2_minus_3v2", 0.0)
    phi += S * c * (w * (2.0 * w * w - 3.0 * u * u - 3.0 * v * v))
    Ex += +E0 * c * (6.0 * u * w)
    Ey += +E0 * c * (6.0 * v * w)
    Ez += -E0 * c * (6.0 * w * w - 3.0 * u * u - 3.0 * v * v)


def sample_fourier_terms(seed=0, n_terms=4, scale=0.10):
    rng = np.random.default_rng(seed + 1000)
    terms = []
    for _ in range(n_terms):
        alpha = rng.integers(1, 5)
        beta = rng.integers(1, 5)
        amplitude = rng.normal(0.0, scale)
        phase_u = rng.uniform(0.0, 2.0 * np.pi)
        phase_v = rng.uniform(0.0, 2.0 * np.pi)
        z_parity = "even" if rng.random() < 0.5 else "odd"
        terms.append({
            "amplitude": float(amplitude),
            "alpha": float(alpha),
            "beta": float(beta),
            "phase_u": float(phase_u),
            "phase_v": float(phase_v),
            "z_parity": z_parity
        })
    return terms


def add_fourier_harmonics(phi, Ex, Ey, Ez, u, v, w, terms, E0, a, wmax):
    S = E0 * a

    for term in terms:
        A = term["amplitude"]
        alpha = term["alpha"]
        beta = term["beta"]
        pu = term.get("phase_u", 0.0)
        pv = term.get("phase_v", 0.0)
        gamma = np.sqrt(alpha * alpha + beta * beta)

        cu = np.cos(alpha * u + pu)
        su = np.sin(alpha * u + pu)
        cv = np.cos(beta * v + pv)
        sv = np.sin(beta * v + pv)

        if term.get("z_parity", "even") == "even":
            norm = np.cosh(gamma * wmax)
            hz = np.cosh(gamma * w) / norm
            dhz = np.sinh(gamma * w) / norm
        else:
            norm = np.sinh(gamma * wmax)
            hz = np.sinh(gamma * w) / norm
            dhz = np.cosh(gamma * w) / norm

        phi += S * A * cu * cv * hz
        Ex += +E0 * A * alpha * su * cv * hz
        Ey += +E0 * A * beta * cu * sv * hz
        Ez += -E0 * A * gamma * cu * cv * dhz


def generate_source_free_field(cfg: BoxConfig, poly_coeffs=None, fourier_terms=None):
    x, y, z, X, Y, Z, Xc, Yc, Zc, u, v, w, a, wmax = make_grid(cfg)

    phi = np.zeros_like(X)
    Ex = np.zeros_like(X)
    Ey = np.zeros_like(X)
    Ez = np.zeros_like(X)

    used_poly_coeffs = {}
    used_fourier_terms = []

    if cfg.use_polynomial:
        used_poly_coeffs = poly_coeffs if poly_coeffs is not None else sample_poly_coefficients(
            seed=cfg.seed,
            scale=cfg.poly_scale
        )
        add_polynomial_harmonics(phi, Ex, Ey, Ez, u, v, w, used_poly_coeffs, cfg.E0, a)

    if cfg.use_fourier:
        used_fourier_terms = fourier_terms if fourier_terms is not None else sample_fourier_terms(
            seed=cfg.seed,
            n_terms=cfg.n_fourier_terms,
            scale=cfg.fourier_scale
        )
        add_fourier_harmonics(phi, Ex, Ey, Ez, u, v, w, used_fourier_terms, cfg.E0, a, wmax)

    E = np.stack([Ex, Ey, Ez], axis=0)
    grid = np.stack([X, Y, Z], axis=0)

    return {
        "config": cfg,
        "grid": grid,
        "x": x,
        "y": y,
        "z": z,
        "phi": phi,
        "E": E,
        "poly_coefficients": used_poly_coeffs,
        "fourier_terms": used_fourier_terms,
    }


def extract_boundary_data(field_dict):
    phi = field_dict["phi"]
    Ex, Ey, Ez = field_dict["E"]

    boundary = {
        "phi_xmin": phi[0, :, :].copy(),
        "phi_xmax": phi[-1, :, :].copy(),
        "phi_ymin": phi[:, 0, :].copy(),
        "phi_ymax": phi[:, -1, :].copy(),
        "phi_zmin": phi[:, :, 0].copy(),
        "phi_zmax": phi[:, :, -1].copy(),
        "Eout_xmin": (-Ex[0, :, :]).copy(),
        "Eout_xmax": (+Ex[-1, :, :]).copy(),
        "Eout_ymin": (-Ey[:, 0, :]).copy(),
        "Eout_ymax": (+Ey[:, -1, :]).copy(),
        "Eout_zmin": (-Ez[:, :, 0]).copy(),
        "Eout_zmax": (+Ez[:, :, -1]).copy(),
    }

    boundary["sigma_xmin"] = EPS0 * boundary["Eout_xmin"]
    boundary["sigma_xmax"] = EPS0 * boundary["Eout_xmax"]
    boundary["sigma_ymin"] = EPS0 * boundary["Eout_ymin"]
    boundary["sigma_ymax"] = EPS0 * boundary["Eout_ymax"]
    boundary["sigma_zmin"] = EPS0 * boundary["Eout_zmin"]
    boundary["sigma_zmax"] = EPS0 * boundary["Eout_zmax"]

    return boundary


def numerical_diagnostics(field_dict):
    x = field_dict["x"]
    y = field_dict["y"]
    z = field_dict["z"]
    Ex, Ey, Ez = field_dict["E"]

    dEx_dx = np.gradient(Ex, x, axis=0, edge_order=2)
    dEy_dy = np.gradient(Ey, y, axis=1, edge_order=2)
    dEz_dz = np.gradient(Ez, z, axis=2, edge_order=2)
    divE = dEx_dx + dEy_dy + dEz_dz

    dEz_dy = np.gradient(Ez, y, axis=1, edge_order=2)
    dEy_dz = np.gradient(Ey, z, axis=2, edge_order=2)
    dEx_dz = np.gradient(Ex, z, axis=2, edge_order=2)
    dEz_dx = np.gradient(Ez, x, axis=0, edge_order=2)
    dEy_dx = np.gradient(Ey, x, axis=0, edge_order=2)
    dEx_dy = np.gradient(Ex, y, axis=1, edge_order=2)

    curl_x = dEz_dy - dEy_dz
    curl_y = dEx_dz - dEz_dx
    curl_z = dEy_dx - dEx_dy
    curl_mag = np.sqrt(curl_x * curl_x + curl_y * curl_y + curl_z * curl_z)

    return {
        "div_rms": float(np.sqrt(np.mean(divE * divE))),
        "div_max_abs": float(np.max(np.abs(divE))),
        "curl_rms": float(np.sqrt(np.mean(curl_mag * curl_mag))),
        "curl_max_abs": float(np.max(np.abs(curl_mag))),
    }


def save_field_npz(path, field_dict, boundary_dict=None):
    save_dict = {
        "grid": field_dict["grid"],
        "x": field_dict["x"],
        "y": field_dict["y"],
        "z": field_dict["z"],
        "phi": field_dict["phi"],
        "E": field_dict["E"],
    }

    if boundary_dict is not None:
        for k, v in boundary_dict.items():
            save_dict[k] = v

    np.savez_compressed(path, **save_dict)


def _downsample_field(field_dict, stride=3):
    X, Y, Z = field_dict["grid"]
    Ex, Ey, Ez = field_dict["E"]

    return (
        X[::stride, ::stride, ::stride],
        Y[::stride, ::stride, ::stride],
        Z[::stride, ::stride, ::stride],
        Ex[::stride, ::stride, ::stride],
        Ey[::stride, ::stride, ::stride],
        Ez[::stride, ::stride, ::stride],
    )


def plot_3d_field_cone(field_dict, stride=4, sizeref=0.35, title="3차원 전기장 벡터"):
    Xs, Ys, Zs, Exs, Eys, Ezs = _downsample_field(field_dict, stride=stride)

    fig = go.Figure(
        data=go.Cone(
            x=Xs.ravel(),
            y=Ys.ravel(),
            z=Zs.ravel(),
            u=Exs.ravel(),
            v=Eys.ravel(),
            w=Ezs.ravel(),
            sizemode="scaled",
            sizeref=sizeref,
            showscale=True
        )
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x 좌표 (m)",
            yaxis_title="y 좌표 (m)",
            zaxis_title="z 좌표 (m)",
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()
    return fig


def plot_3d_equipotential(field_dict, stride=2, surface_count=7, opacity=0.22, title="3차원 등전위면"):
    X, Y, Z = field_dict["grid"]
    phi = field_dict["phi"]

    Xs = X[::stride, ::stride, ::stride]
    Ys = Y[::stride, ::stride, ::stride]
    Zs = Z[::stride, ::stride, ::stride]
    Ps = phi[::stride, ::stride, ::stride]

    pmin = float(np.percentile(Ps, 12))
    pmax = float(np.percentile(Ps, 88))

    fig = go.Figure(
        data=go.Isosurface(
            x=Xs.ravel(),
            y=Ys.ravel(),
            z=Zs.ravel(),
            value=Ps.ravel(),
            isomin=pmin,
            isomax=pmax,
            surface_count=surface_count,
            opacity=opacity,
            caps=dict(x_show=False, y_show=False, z_show=False),
            showscale=True
        )
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x 좌표 (m)",
            yaxis_title="y 좌표 (m)",
            zaxis_title="z 좌표 (m)",
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()
    return fig


def plot_2d_slices(field_dict, x_index=None, y_index=None, z_index=None, figsize=(15, 4)):
    x = field_dict["x"]
    y = field_dict["y"]
    z = field_dict["z"]
    phi = field_dict["phi"]
    Ex, Ey, Ez = field_dict["E"]
    Emag = np.sqrt(Ex * Ex + Ey * Ey + Ez * Ez)

    if x_index is None:
        x_index = len(x) // 2
    if y_index is None:
        y_index = len(y) // 2
    if z_index is None:
        z_index = len(z) // 2

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    c0 = axes[0].contourf(y, z, phi[x_index, :, :].T, levels=40)
    axes[0].set_title(f"x = {x[x_index]:.3f} m 단면의 전위 φ")
    axes[0].set_xlabel("y 좌표 (m)")
    axes[0].set_ylabel("z 좌표 (m)")
    plt.colorbar(c0, ax=axes[0])

    c1 = axes[1].contourf(x, z, phi[:, y_index, :].T, levels=40)
    axes[1].set_title(f"y = {y[y_index]:.3f} m 단면의 전위 φ")
    axes[1].set_xlabel("x 좌표 (m)")
    axes[1].set_ylabel("z 좌표 (m)")
    plt.colorbar(c1, ax=axes[1])

    c2 = axes[2].contourf(x, y, Emag[:, :, z_index].T, levels=40)
    axes[2].set_title(f"z = {z[z_index]:.3f} m 단면의 전기장 세기 |E|")
    axes[2].set_xlabel("x 좌표 (m)")
    axes[2].set_ylabel("y 좌표 (m)")
    plt.colorbar(c2, ax=axes[2])

    plt.tight_layout()
    plt.show()


def summarize_field(field_dict):
    report = numerical_diagnostics(field_dict)
    phi = field_dict["phi"]
    Ex, Ey, Ez = field_dict["E"]
    Emag = np.sqrt(Ex * Ex + Ey * Ey + Ez * Ez)

    summary = {
        "phi_min": float(phi.min()),
        "phi_max": float(phi.max()),
        "E_min": float(Emag.min()),
        "E_max": float(Emag.max()),
        "E_mean": float(Emag.mean()),
        **report
    }

    return summary


In [ ]:
# 목표 전기장 생성
cfg = BoxConfig(
    Lx=1.30,          # Box length in x
    Ly=0.85,          # Box length in y
    Lz=1.10,          # Box length in z
    Nx=65,            # Grid points in x
    Ny=51,            # Grid points in y
    Nz=57,            # Grid points in z
    E0=1.0,           # Global field scale
    seed=12,          # Change this to get a different random field
    poly_scale=0.18,  # Increase for stronger polynomial distortion
    fourier_scale=0.12,  # Increase for richer oscillatory structure
    n_fourier_terms=5,   # Increase to mix more spatial modes
    use_polynomial=True, # Turn polynomial family on or off
    use_fourier=True     # Turn Fourier harmonic family on or off
)

poly_coeffs = {
    "u": 0.55,                      # Main uniform x-directed component
    "v": -0.30,                     # Main uniform y-directed component
    "w": 0.20,                      # Main uniform z-directed component
    "uv": 0.08,                     # Adds x-y coupling
    "uw": -0.04,                    # Adds x-z coupling
    "vw": 0.06,                     # Adds y-z coupling
    "u2_minus_v2": 0.10,            # Adds quadrupole-like structure
    "2w2_minus_u2_minus_v2": -0.08, # Adds axial anisotropy
    "u_4w2_u2_v2": 0.03,            # Adds higher-order x variation
    "v_4w2_u2_v2": -0.02,           # Adds higher-order y variation
    "w_u2_minus_v2": 0.05,          # Adds mixed z-anisotropy
    "uvw": 0.04,                    # Adds fully mixed cubic variation
    "u_u2_minus_3v2": -0.03,        # Adds planar harmonic asymmetry
    "v_3u2_minus_v2": 0.02,         # Adds another planar asymmetry
    "w_2w2_minus_3u2_minus_3v2": 0.01 # Adds higher-order z asymmetry
}

fourier_terms = [
    {
        "amplitude": 0.10,   # Increase amplitude for stronger oscillation
        "alpha": 2.0,        # Increase to make finer variation along x
        "beta": 1.0,         # Increase to make finer variation along y
        "phase_u": 0.0,      # Shift phase along x
        "phase_v": 0.7,      # Shift phase along y
        "z_parity": "even"   # Use "even" or "odd" to change z symmetry
    },
    {
        "amplitude": -0.07,  # Negative amplitude flips the mode contribution
        "alpha": 3.0,        # Larger alpha makes more x oscillations
        "beta": 2.0,         # Larger beta makes more y oscillations
        "phase_u": 0.8,      # Change to move the pattern in x
        "phase_v": 1.6,      # Change to move the pattern in y
        "z_parity": "odd"    # Odd parity changes how the field varies across z=0
    },
    {
        "amplitude": 0.05,   # Add or remove terms here to diversify the field family
        "alpha": 1.0,
        "beta": 4.0,
        "phase_u": 1.2,
        "phase_v": 0.2,
        "z_parity": "even"
    }
]

field = generate_source_free_field(cfg, poly_coeffs=poly_coeffs, fourier_terms=fourier_terms)
boundary = extract_boundary_data(field)
summary = summarize_field(field)

summary


In [ ]:
# 표면전하 역산 함수 정의
import numba
from numba import prange
import time

# =============================================================================
#  Constants
# =============================================================================
_K_COULOMB = 1.0 / (4.0 * np.pi * EPS0)

try:
    import cupy as cp
    HAS_CUPY = True
    print("CuPy detected — GPU-accelerated least-squares available.")
except ImportError:
    HAS_CUPY = False
    print("CuPy not found — using CPU (NumPy). Install cupy for GPU acceleration.")

# =============================================================================
#  Numba-accelerated Green's function matrix
# =============================================================================

@numba.njit(parallel=True, cache=False, fastmath=True)
def _build_green_E_numba(eval_pts, centers, areas, soft2, k):
    """Green's matrix G: E_flat(3N,) = G(3N, M) @ sigma(M,)."""
    N = eval_pts.shape[0]
    M = centers.shape[0]
    G = np.zeros((3 * N, M))
    for i in prange(N):
        px = eval_pts[i, 0]
        py = eval_pts[i, 1]
        pz = eval_pts[i, 2]
        row0 = 3 * i
        row1 = 3 * i + 1
        row2 = 3 * i + 2
        for j in range(M):
            dx = px - centers[j, 0]
            dy = py - centers[j, 1]
            dz = pz - centers[j, 2]
            r2 = dx * dx + dy * dy + dz * dz + soft2
            inv_r = 1.0 / np.sqrt(r2)
            factor = k * areas[j] * inv_r / r2
            G[row0, j] = factor * dx
            G[row1, j] = factor * dy
            G[row2, j] = factor * dz
    return G


@numba.njit(parallel=True, cache=False, fastmath=True)
def _build_green_phi_numba(eval_pts, centers, areas, soft2, k):
    """Green's matrix G_phi: phi(N,) = G_phi(N, M) @ sigma(M,)."""
    N = eval_pts.shape[0]
    M = centers.shape[0]
    G = np.zeros((N, M))
    for i in prange(N):
        px = eval_pts[i, 0]
        py = eval_pts[i, 1]
        pz = eval_pts[i, 2]
        for j in range(M):
            dx = px - centers[j, 0]
            dy = py - centers[j, 1]
            dz = pz - centers[j, 2]
            r2 = dx * dx + dy * dy + dz * dz + soft2
            G[i, j] = k * areas[j] / np.sqrt(r2)
    return G


def _to_f64_contiguous(arr):
    return np.ascontiguousarray(arr, dtype=np.float64)


def build_green_matrices(eval_points, patch_centers, patch_areas, softening=0.0):
    """Build Green's function matrices for E and phi."""
    soft2 = softening ** 2
    pts = _to_f64_contiguous(eval_points)
    cen = _to_f64_contiguous(patch_centers)
    area = _to_f64_contiguous(patch_areas)
    G_E = _build_green_E_numba(pts, cen, area, soft2, _K_COULOMB)
    G_phi = _build_green_phi_numba(pts, cen, area, soft2, _K_COULOMB)
    return G_E, G_phi


def build_green_E_only(eval_points, patch_centers, patch_areas, softening=0.0):
    """Build Green's matrix for E field only (faster when phi not needed)."""
    soft2 = softening ** 2
    return _build_green_E_numba(
        _to_f64_contiguous(eval_points),
        _to_f64_contiguous(patch_centers),
        _to_f64_contiguous(patch_areas),
        soft2, _K_COULOMB,
    )


# =============================================================================
#  Error metrics
# =============================================================================

def compare_target_and_reconstructed_fields(target_E, reconstructed_E,
                                            target_phi=None, reconstructed_phi=None):
    """Compute comprehensive error metrics between target and reconstructed fields."""
    target_E = np.asarray(target_E, dtype=float)
    reconstructed_E = np.asarray(reconstructed_E, dtype=float)

    dE = reconstructed_E - target_E
    target_mag = np.linalg.norm(target_E, axis=1)
    rec_mag = np.linalg.norm(reconstructed_E, axis=1)
    err_mag = np.linalg.norm(dE, axis=1)

    denom = np.maximum(target_mag, 1e-15)
    rel_point_error = err_mag / denom

    dot = np.sum(target_E * reconstructed_E, axis=1)
    cosine = dot / np.maximum(target_mag * rec_mag, 1e-15)

    out = {
        "E_component_rmse": float(np.sqrt(np.mean(dE * dE))),
        "E_vector_rmse": float(np.sqrt(np.mean(err_mag ** 2))),
        "E_vector_mae": float(np.mean(err_mag)),
        "E_relative_l2": float(np.linalg.norm(dE) / max(np.linalg.norm(target_E), 1e-15)),
        "E_relative_mae_mean": float(np.mean(rel_point_error)),
        "E_relative_mae_median": float(np.median(rel_point_error)),
        "E_relative_mae_p95": float(np.percentile(rel_point_error, 95)),
        "target_E_mean": float(np.mean(target_mag)),
        "target_E_max": float(np.max(target_mag)),
        "reconstructed_E_mean": float(np.mean(rec_mag)),
        "reconstructed_E_max": float(np.max(rec_mag)),
        "mean_cosine_similarity": float(np.mean(cosine)),
    }

    if target_phi is not None and reconstructed_phi is not None:
        target_phi = np.asarray(target_phi, dtype=float)
        reconstructed_phi = np.asarray(reconstructed_phi, dtype=float)
        dphi = reconstructed_phi - target_phi
        out["phi_rmse_raw"] = float(np.sqrt(np.mean(dphi * dphi)))
        out["phi_relative_l2_raw"] = float(
            np.linalg.norm(dphi) / max(np.linalg.norm(target_phi), 1e-15))

        dphi_c = (reconstructed_phi - reconstructed_phi.mean()) - (target_phi - target_phi.mean())
        target_phi_c = target_phi - target_phi.mean()
        out["phi_rmse_centered"] = float(np.sqrt(np.mean(dphi_c ** 2)))
        out["phi_relative_l2_centered"] = float(
            np.linalg.norm(dphi_c) / max(np.linalg.norm(target_phi_c), 1e-15))

    return out


# =============================================================================
#  Interior sampling
# =============================================================================

def sample_interior(field_dict, sample_stride=4, interior_margin=1):
    """Sample interior target E and phi on a sub-grid."""
    x, y, z = field_dict["x"], field_dict["y"], field_dict["z"]
    Ex, Ey, Ez = field_dict["E"]
    phi = field_dict["phi"]

    m = interior_margin
    xs = x[m:len(x) - m:sample_stride]
    ys = y[m:len(y) - m:sample_stride]
    zs = z[m:len(z) - m:sample_stride]

    sl = (
        slice(m, len(x) - m, sample_stride),
        slice(m, len(y) - m, sample_stride),
        slice(m, len(z) - m, sample_stride),
    )

    Xs, Ys, Zs = np.meshgrid(xs, ys, zs, indexing="ij")

    return {
        "points": np.column_stack([Xs.ravel(), Ys.ravel(), Zs.ravel()]),
        "target_E": np.column_stack([Ex[sl].ravel(), Ey[sl].ravel(), Ez[sl].ravel()]),
        "target_phi": phi[sl].ravel(),
        "x": xs, "y": ys, "z": zs,
        "shape": (len(xs), len(ys), len(zs)),
    }


# =============================================================================
#  Patch builders
# =============================================================================

def trapezoid_weights(coords):
    coords = np.asarray(coords, dtype=float)
    w = np.empty_like(coords)
    diffs = np.diff(coords)
    w[0] = 0.5 * diffs[0]
    w[-1] = 0.5 * diffs[-1]
    if coords.size > 2:
        w[1:-1] = 0.5 * (diffs[:-1] + diffs[1:])
    return w


def build_continuous_patches(field_dict, surface_stride=1):
    """Build boundary patches at grid resolution (continuous approximation)."""
    x, y, z = field_dict["x"], field_dict["y"], field_dict["z"]

    ys = y[::surface_stride]
    zs = z[::surface_stride]
    xs = x[::surface_stride]

    wy = trapezoid_weights(ys)
    wz = trapezoid_weights(zs)
    wx = trapezoid_weights(xs)

    centers_all = []
    areas_all = []
    face_all = []

    area_yz = np.outer(wy, wz)
    for face_name, xval in [("xmin", x[0]), ("xmax", x[-1])]:
        Yf, Zf = np.meshgrid(ys, zs, indexing="ij")
        Xf = np.full_like(Yf, xval)
        centers_all.append(np.column_stack([Xf.ravel(), Yf.ravel(), Zf.ravel()]))
        areas_all.append(area_yz.ravel().copy())
        face_all.append(np.full(Yf.size, face_name))

    area_xz = np.outer(wx, wz)
    for face_name, yval in [("ymin", y[0]), ("ymax", y[-1])]:
        Xf, Zf = np.meshgrid(xs, zs, indexing="ij")
        Yf = np.full_like(Xf, yval)
        centers_all.append(np.column_stack([Xf.ravel(), Yf.ravel(), Zf.ravel()]))
        areas_all.append(area_xz.ravel().copy())
        face_all.append(np.full(Xf.size, face_name))

    area_xy = np.outer(wx, wy)
    for face_name, zval in [("zmin", z[0]), ("zmax", z[-1])]:
        Xf, Yf = np.meshgrid(xs, ys, indexing="ij")
        Zf = np.full_like(Xf, zval)
        centers_all.append(np.column_stack([Xf.ravel(), Yf.ravel(), Zf.ravel()]))
        areas_all.append(area_xy.ravel().copy())
        face_all.append(np.full(Xf.size, face_name))

    return {
        "centers": np.vstack(centers_all),
        "areas": np.concatenate(areas_all),
        "face": np.concatenate(face_all),
        "num_patches": sum(a.size for a in areas_all),
        "surface_stride": surface_stride,
    }


def build_discrete_nm_patches(field_dict, n, m):
    """Build n×m uniform rectangular patches on each of the 6 box faces."""
    x, y, z = field_dict["x"], field_dict["y"], field_dict["z"]
    x0, x1 = float(x[0]), float(x[-1])
    y0, y1 = float(y[0]), float(y[-1])
    z0, z1 = float(z[0]), float(z[-1])

    face_specs = [
        ("xmin", 0, x0, (y0, y1), (z0, z1)),
        ("xmax", 0, x1, (y0, y1), (z0, z1)),
        ("ymin", 1, y0, (x0, x1), (z0, z1)),
        ("ymax", 1, y1, (x0, x1), (z0, z1)),
        ("zmin", 2, z0, (x0, x1), (y0, y1)),
        ("zmax", 2, z1, (x0, x1), (y0, y1)),
    ]

    total = 6 * n * m
    centers = np.empty((total, 3))
    areas = np.empty(total)
    face = np.empty(total, dtype=object)

    idx = 0
    for face_name, axis, fixed_val, (d1s, d1e), (d2s, d2e) in face_specs:
        d1_size = (d1e - d1s) / n
        d2_size = (d2e - d2s) / m
        area = d1_size * d2_size
        for i in range(n):
            d1_c = d1s + (i + 0.5) * d1_size
            for j in range(m):
                d2_c = d2s + (j + 0.5) * d2_size
                if axis == 0:
                    centers[idx] = [fixed_val, d1_c, d2_c]
                elif axis == 1:
                    centers[idx] = [d1_c, fixed_val, d2_c]
                else:
                    centers[idx] = [d1_c, d2_c, fixed_val]
                areas[idx] = area
                face[idx] = face_name
                idx += 1

    return {
        "centers": centers,
        "areas": areas,
        "face": face,
        "n": n, "m": m,
        "num_patches": total,
    }


# =============================================================================
#  Least-squares solver  (normal equations: faster for M << 3N)
# =============================================================================

def solve_lstsq(G, target, regularization=0.0, use_gpu=False):
    """Solve  min ||target - G @ σ||² + λ||σ||²  via normal equations.

    Forms G^T G (M×M) and solves  (G^T G + λI) σ = G^T b.
    This is O(N·M²) — much faster than full lstsq when N >> M.
    """
    if use_gpu and HAS_CUPY:
        G_d = cp.asarray(G)
        b_d = cp.asarray(target)
        GtG = G_d.T @ G_d
        Gtb = G_d.T @ b_d
        if regularization > 0.0:
            GtG += regularization * cp.eye(GtG.shape[0], dtype=GtG.dtype)
        sigma = cp.linalg.solve(GtG, Gtb)
        return cp.asnumpy(sigma)
    else:
        GtG = G.T @ G
        Gtb = G.T @ target
        if regularization > 0.0:
            GtG += regularization * np.eye(GtG.shape[0])
        return np.linalg.solve(GtG, Gtb)


# =============================================================================
#  1) Continuous charge-distribution inversion
# =============================================================================

def _auto_softening(field_dict):
    dx = np.min(np.diff(field_dict["x"]))
    dy = np.min(np.diff(field_dict["y"]))
    dz = np.min(np.diff(field_dict["z"]))
    return 0.35 * min(dx, dy, dz)


def run_continuous_inversion(field_dict, surface_stride=2, sample_stride=4,
                              interior_margin=2, softening=None,
                              regularization=1e-12, use_gpu=False):
    """Reconstruct continuous surface-charge σ(r) via least-squares."""
    t0 = time.time()

    if softening is None:
        softening = _auto_softening(field_dict)

    samples = sample_interior(field_dict, sample_stride, interior_margin)
    patches = build_continuous_patches(field_dict, surface_stride)

    print(f"  Building Green's matrix ({3*samples['points'].shape[0]} × {patches['num_patches']}) ...")
    G_E, G_phi = build_green_matrices(
        samples["points"], patches["centers"], patches["areas"], softening,
    )

    target_E_flat = samples["target_E"].ravel()
    sigma_opt = solve_lstsq(G_E, target_E_flat, regularization, use_gpu)

    rec_E_flat = G_E @ sigma_opt
    rec_phi = G_phi @ sigma_opt
    rec_E = rec_E_flat.reshape(-1, 3)

    metrics = compare_target_and_reconstructed_fields(
        samples["target_E"], rec_E,
        target_phi=samples["target_phi"],
        reconstructed_phi=rec_phi,
    )
    elapsed = time.time() - t0
    metrics.update({
        "num_patches": patches["num_patches"],
        "num_eval_points": samples["points"].shape[0],
        "surface_stride": surface_stride,
        "softening": softening,
        "regularization": regularization,
        "elapsed_sec": elapsed,
    })

    return {
        "patches": patches,
        "samples": samples,
        "sigma_opt": sigma_opt,
        "reconstructed_E": rec_E,
        "reconstructed_phi": rec_phi,
        "metrics": metrics,
    }


# =============================================================================
#  2) Discrete n×m charge-distribution inversion
# =============================================================================

def run_discrete_nm_inversion(field_dict, n, m, sample_stride=4,
                               interior_margin=2, softening=None,
                               regularization=0.0, use_gpu=False):
    """Find optimal uniform σ on 6·n·m patches (least-squares)."""
    t0 = time.time()

    if softening is None:
        softening = _auto_softening(field_dict)

    samples = sample_interior(field_dict, sample_stride, interior_margin)
    patches = build_discrete_nm_patches(field_dict, n, m)

    G_E, G_phi = build_green_matrices(
        samples["points"], patches["centers"], patches["areas"], softening,
    )

    target_E_flat = samples["target_E"].ravel()
    sigma_opt = solve_lstsq(G_E, target_E_flat, regularization, use_gpu)

    rec_E = (G_E @ sigma_opt).reshape(-1, 3)
    rec_phi = G_phi @ sigma_opt

    metrics = compare_target_and_reconstructed_fields(
        samples["target_E"], rec_E,
        target_phi=samples["target_phi"],
        reconstructed_phi=rec_phi,
    )
    elapsed = time.time() - t0

    face_charge = {}
    for fn in ["xmin", "xmax", "ymin", "ymax", "zmin", "zmax"]:
        mask = patches["face"] == fn
        face_charge[fn] = float(np.sum(sigma_opt[mask] * patches["areas"][mask]))

    metrics.update({
        "n": n, "m": m,
        "num_patches": patches["num_patches"],
        "num_eval_points": samples["points"].shape[0],
        "softening": softening,
        "regularization": regularization,
        "elapsed_sec": elapsed,
    })

    return {
        "patches": patches,
        "samples": samples,
        "sigma_opt": sigma_opt,
        "reconstructed_E": rec_E,
        "reconstructed_phi": rec_phi,
        "metrics": metrics,
        "face_charge": face_charge,
    }


# =============================================================================
#  3) Sweep (n, m) to find optimal grid resolution
# =============================================================================

def sweep_nm_optimal(field_dict, n_values, m_values=None,
                      sample_stride=4, interior_margin=2,
                      softening=None, regularization=0.0,
                      use_gpu=False, verbose=True):
    """Sweep (n, m) pairs — find grid minimising rel-L2 error."""
    if m_values is None:
        m_values = n_values

    if softening is None:
        softening = _auto_softening(field_dict)

    # Pre-compute samples once (shared across all (n,m))
    samples = sample_interior(field_dict, sample_stride, interior_margin)
    target_E_flat = samples["target_E"].ravel()
    target_E_norm = np.linalg.norm(target_E_flat)

    results = []
    best_error = np.inf
    best_nm = (None, None)

    total = len(n_values) * len(m_values)
    count = 0
    t_total = time.time()

    for ni in n_values:
        for mi in m_values:
            count += 1
            t0 = time.time()

            patches = build_discrete_nm_patches(field_dict, ni, mi)
            G_E = build_green_E_only(
                samples["points"], patches["centers"], patches["areas"], softening,
            )
            sigma_opt = solve_lstsq(G_E, target_E_flat, regularization, use_gpu)

            rec_E_flat = G_E @ sigma_opt
            dE = rec_E_flat - target_E_flat
            rel_l2 = float(np.linalg.norm(dE) / max(target_E_norm, 1e-15))

            rec_E = rec_E_flat.reshape(-1, 3)
            target_mag = np.linalg.norm(samples["target_E"], axis=1)
            rec_mag = np.linalg.norm(rec_E, axis=1)
            mean_cos = float(np.mean(
                np.sum(samples["target_E"] * rec_E, axis=1)
                / np.maximum(target_mag * rec_mag, 1e-15)
            ))
            rmse = float(np.sqrt(np.mean(np.sum((rec_E - samples["target_E"])**2, axis=1))))

            dt = time.time() - t0

            entry = {
                "n": ni, "m": mi,
                "num_patches": 6 * ni * mi,
                "E_relative_l2": rel_l2,
                "E_rmse": rmse,
                "mean_cosine": mean_cos,
                "elapsed_sec": dt,
            }
            results.append(entry)

            if rel_l2 < best_error:
                best_error = rel_l2
                best_nm = (ni, mi)

            if verbose:
                print(f"  [{count:3d}/{total}] n={ni:2d}, m={mi:2d} | "
                      f"patches={6*ni*mi:5d} | rel_L2={rel_l2:.6f} | "
                      f"cos={mean_cos:.6f} | {dt:.2f}s")

    total_time = time.time() - t_total
    if verbose:
        print(f"\nSweep complete in {total_time:.1f}s.")
        print(f"Optimal: n={best_nm[0]}, m={best_nm[1]} "
              f"(rel_L2 = {best_error:.6f}, patches = {6*best_nm[0]*best_nm[1]})")

    return {
        "results": results,
        "best_n": best_nm[0],
        "best_m": best_nm[1],
        "best_error": best_error,
        "n_values": list(n_values),
        "m_values": list(m_values),
    }


# =============================================================================
#  Visualization helpers
# =============================================================================

def plot_reconstruction_comparison(result, component="mag"):
    """Compare target vs reconstructed E on a z-mid slice."""
    xs, ys, zs = result["samples"]["x"], result["samples"]["y"], result["samples"]["z"]
    shape = result["samples"]["shape"]
    target_E = result["samples"]["target_E"].reshape(*shape, 3)
    rec_E = result["reconstructed_E"].reshape(*shape, 3)

    z_idx = len(zs) // 2
    if component == "mag":
        target = np.linalg.norm(target_E[:, :, z_idx, :], axis=2)
        recon = np.linalg.norm(rec_E[:, :, z_idx, :], axis=2)
        label = "|E|"
    else:
        ci = {"x": 0, "y": 1, "z": 2}[component]
        target = target_E[:, :, z_idx, ci]
        recon = rec_E[:, :, z_idx, ci]
        label = f"E_{component}"

    error = recon - target

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    c0 = axes[0].contourf(xs, ys, target.T, levels=30)
    axes[0].set_title(f"목표 {label}  (z = {zs[z_idx]:.3f} m)")
    axes[0].set_xlabel("x 좌표 (m)"); axes[0].set_ylabel("y 좌표 (m)")
    plt.colorbar(c0, ax=axes[0])

    c1 = axes[1].contourf(xs, ys, recon.T, levels=30)
    axes[1].set_title(f"재구성 {label}")
    axes[1].set_xlabel("x 좌표 (m)"); axes[1].set_ylabel("y 좌표 (m)")
    plt.colorbar(c1, ax=axes[1])

    c2 = axes[2].contourf(xs, ys, error.T, levels=30, cmap="RdBu_r")
    axes[2].set_title(f"오차 ({label})")
    axes[2].set_xlabel("x 좌표 (m)"); axes[2].set_ylabel("y 좌표 (m)")
    plt.colorbar(c2, ax=axes[2])

    patches_info = result.get("patches", {})
    if "surface_stride" in patches_info and "n" not in patches_info:
        nm_label = "연속 역산"
    else:
        nm_label = f"n={result['metrics'].get('n','?')}, m={result['metrics'].get('m','?')}"
    plt.suptitle(f"{nm_label}  |  상대 L2 오차 = {result['metrics']['E_relative_l2']:.6f}", fontsize=11)
    plt.tight_layout()
    plt.show()


def plot_parity(result, max_points=3000):
    """Parity plot of target vs reconstructed |E|."""
    target_mag = np.linalg.norm(result["samples"]["target_E"], axis=1)
    rec_mag = np.linalg.norm(result["reconstructed_E"], axis=1)
    n = target_mag.size
    if n > max_points:
        idx = np.linspace(0, n - 1, max_points, dtype=int)
        target_mag, rec_mag = target_mag[idx], rec_mag[idx]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(target_mag, rec_mag, s=10, alpha=0.5)
    lo, hi = min(target_mag.min(), rec_mag.min()), max(target_mag.max(), rec_mag.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("목표 전기장 세기 |E| (V/m)"); ax.set_ylabel("재구성 전기장 세기 |E| (V/m)")
    ax.set_title("패리티 플롯 (목표 대 재구성 비교)")
    plt.tight_layout(); plt.show()


def plot_sigma_distribution(result, face="xmin"):
    """Visualise the reconstructed σ on one face of the box."""
    patches = result["patches"]
    sigma = result["sigma_opt"]
    mask = patches["face"] == face
    centers_f = patches["centers"][mask]
    sigma_f = sigma[mask]
    n, m = patches["n"], patches["m"]

    if face.startswith("x"):
        a1, a2, l1, l2 = 1, 2, "y", "z"
    elif face.startswith("y"):
        a1, a2, l1, l2 = 0, 2, "x", "z"
    else:
        a1, a2, l1, l2 = 0, 1, "x", "y"

    sigma_grid = sigma_f.reshape(n, m)
    c1 = centers_f[:, a1].reshape(n, m)
    c2 = centers_f[:, a2].reshape(n, m)

    fig, ax = plt.subplots(figsize=(7, 5))
    vabs = max(abs(sigma_grid.min()), abs(sigma_grid.max()))
    if vabs < 1e-30:
        vabs = 1.0
    pcm = ax.pcolormesh(c1, c2, sigma_grid, shading="auto",
                        cmap="RdBu_r", vmin=-vabs, vmax=vabs)
    ax.set_xlabel(f"{l1} 좌표 (m)"); ax.set_ylabel(f"{l2} 좌표 (m)")
    ax.set_title(f"{face} 면 재구성 전하 밀도 σ  (n={n}, m={m})")
    ax.set_aspect("equal")
    plt.colorbar(pcm, ax=ax, label="전하 밀도 σ [C/m²]")
    plt.tight_layout(); plt.show()


def plot_nm_sweep_heatmap(sweep_result, metric="E_relative_l2"):
    """Heatmap of reconstruction error over the (n, m) grid."""
    n_vals = sorted(set(r["n"] for r in sweep_result["results"]))
    m_vals = sorted(set(r["m"] for r in sweep_result["results"]))
    Z = np.full((len(m_vals), len(n_vals)), np.nan)
    for r in sweep_result["results"]:
        i = n_vals.index(r["n"])
        j = m_vals.index(r["m"])
        Z[j, i] = r[metric]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(Z, origin="lower", aspect="auto",
                   extent=[n_vals[0] - 0.5, n_vals[-1] + 0.5,
                           m_vals[0] - 0.5, m_vals[-1] + 0.5])
    ax.set_xlabel("패치 수 n"); ax.set_ylabel("패치 수 m")
    ax.set_title(f"(n, m) 조합에 따른 재구성 오차 ({metric})")
    plt.colorbar(im, ax=ax, label=metric)

    bn, bm = sweep_result["best_n"], sweep_result["best_m"]
    ax.plot(bn, bm, "r*", markersize=15, label=f"최적: n={bn}, m={bm}")
    ax.legend()
    plt.tight_layout(); plt.show()


def plot_nm_sweep_curve(sweep_result, metric="E_relative_l2"):
    """Error vs total patch count (log-scale)."""
    rs = sweep_result["results"]
    patches = [r["num_patches"] for r in rs]
    errors = [r[metric] for r in rs]
    idx = np.argsort(patches)
    patches = [patches[i] for i in idx]
    errors = [errors[i] for i in idx]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(patches, errors, "o-", ms=5)
    ax.set_xlabel("전체 패치 수 (6·n·m)"); ax.set_ylabel(metric)
    ax.set_title("격자 해상도에 따른 재구성 오차")
    ax.set_yscale("log"); ax.grid(True, alpha=0.3)

    bp = 6 * sweep_result["best_n"] * sweep_result["best_m"]
    be = sweep_result["best_error"]
    ax.axvline(bp, color="r", ls="--", alpha=0.5)
    ax.annotate(f"최적: n={sweep_result['best_n']}, m={sweep_result['best_m']}\n{metric}={be:.6f}",
                xy=(bp, be), xytext=(bp + 50, be * 2),
                arrowprops=dict(arrowstyle="->"), fontsize=9)
    plt.tight_layout(); plt.show()


# Warm-up numba JIT (compile once with tiny arrays)
_dummy_pts = np.zeros((2, 3))
_dummy_a = np.ones(2)
_ = _build_green_E_numba(_dummy_pts, _dummy_pts, _dummy_a, 1e-6, _K_COULOMB)
_ = _build_green_phi_numba(_dummy_pts, _dummy_pts, _dummy_a, 1e-6, _K_COULOMB)
print("Numba JIT warm-up complete.")


In [ ]:
# 연속 경계 역산
# =====================================================================
#  STEP 1 — Continuous surface-charge inversion
# =====================================================================
print("=" * 65)
print("STEP 1: Continuous charge-distribution inversion  (fine mesh)")
print("=" * 65)

continuous_result = run_continuous_inversion(
    field,
    surface_stride=2,       # stride=2 on boundary grid → quasi-continuous
    sample_stride=4,        # sparse interior evaluation
    interior_margin=2,      # keep away from boundary singularities
    regularization=1e-12,   # small Tikhonov for numerical stability
)

m = continuous_result["metrics"]
print(f"\n  Boundary patches : {m['num_patches']}")
print(f"  Interior points  : {m['num_eval_points']}")
print(f"  Time             : {m['elapsed_sec']:.2f} s")
print(f"\n  E  relative-L2   : {m['E_relative_l2']:.6f}")
print(f"  E  vector RMSE   : {m['E_vector_rmse']:.6f}")
print(f"  Mean cosine sim. : {m['mean_cosine_similarity']:.6f}")

plot_reconstruction_comparison(continuous_result, "mag")
plot_parity(continuous_result)


In [ ]:
# 10x10 패치 역산
# =====================================================================
#  STEP 2 — Discrete n×m uniform-patch inversion  (single example)
# =====================================================================
print("=" * 65)
print("STEP 2: Discrete n×m charge-distribution inversion")
print("=" * 65)

n_test, m_test = 10, 10
discrete_result = run_discrete_nm_inversion(
    field, n=n_test, m=m_test,
    sample_stride=4,
    interior_margin=2,
)

md = discrete_result["metrics"]
print(f"\n  Grid             : {n_test} × {m_test}  →  {md['num_patches']} patches")
print(f"  Time             : {md['elapsed_sec']:.2f} s")
print(f"\n  E  relative-L2   : {md['E_relative_l2']:.6f}")
print(f"  E  vector RMSE   : {md['E_vector_rmse']:.6f}")
print(f"  Mean cosine sim. : {md['mean_cosine_similarity']:.6f}")

print("\n  Face charges:")
for face_name, q in discrete_result["face_charge"].items():
    print(f"    {face_name}: {q:+.4e} C")

plot_reconstruction_comparison(discrete_result, "mag")
plot_parity(discrete_result)

# σ distribution on selected faces
for face in ["xmin", "xmax", "zmin"]:
    plot_sigma_distribution(discrete_result, face)


In [ ]:
# 패치 해상도 탐색
# =====================================================================
#  STEP 3 — Sweep n, m to find optimal grid resolution
# =====================================================================
print("=" * 65)
print("STEP 3: Optimal (n, m) parameter search")
print("=" * 65)

n_range = list(range(2, 22, 2))      # 2, 4, 6, …, 20
m_range = list(range(2, 22, 2))

sweep = sweep_nm_optimal(
    field,
    n_values=n_range,
    m_values=m_range,
    sample_stride=4,
    interior_margin=2,
    verbose=True,
)

print(f"\n{'=' * 65}")
print(f"  OPTIMAL : n = {sweep['best_n']},  m = {sweep['best_m']}")
print(f"  Patches : {6 * sweep['best_n'] * sweep['best_m']}")
print(f"  rel-L2  : {sweep['best_error']:.6f}")
print(f"{'=' * 65}")

plot_nm_sweep_heatmap(sweep)
plot_nm_sweep_curve(sweep)


In [ ]:
# 20x20 패치 결과 확인
optimal_result = run_discrete_nm_inversion(
    field,
    n=sweep["best_n"],
    m=sweep["best_m"],
    sample_stride=4,
    interior_margin=2,
)

mo = optimal_result["metrics"]
print(f"Optimal n={mo['n']}, m={mo['m']}")
print(f"E relative-L2: {mo['E_relative_l2']:.6f}")
print(f"Mean cosine: {mo['mean_cosine_similarity']:.6f}")

plot_reconstruction_comparison(optimal_result, "mag")


In [ ]:
# 경계 전위 재현 함수 불러오기
from importlib import reload

import generate_field.potential_boundary_reconstruction as pbr

reload(pbr)

from generate_field.potential_boundary_reconstruction import (
    EPS0,
    HAS_CUPY,
    HarmonicWorldConfig,
    SpectralDirichletBoxSolver,
    apply_boundary_noise,
    apply_boundary_patch_approximation,
    boundary_mask,
    compute_electric_field_from_potential,
    compute_laplacian,
    compute_reconstruction_metrics,
    compute_weighted_volume_scores,
    create_harmonic_field_model,
    extract_boundary_potential,
    format_metric_table,
    generate_translation_centers,
    plot_boundary_reconstruction_comparison,
    plot_noise_sweep,
    plot_patch_sweep,
    plot_volume_box_lengths,
    plot_volume_sweep_cloud,
    plot_volume_sweep_summary,
    plot_weighted_volume_score,
    potential_recovery_metrics,
    recover_potential_from_field,
    run_boundary_noise_sweep,
    run_boundary_patch_sweep,
    run_boundary_potential_reconstruction,
    run_volume_translation_sweep,
    sample_fourier_terms,
    sample_model_on_box,
    sample_poly_coefficients,
    trim_mask_by_layers,
 )

world_cfg = HarmonicWorldConfig(
    Lx=cfg.Lx,
    Ly=cfg.Ly,
    Lz=cfg.Lz,
    center=(0.5 * cfg.Lx, 0.5 * cfg.Ly, 0.5 * cfg.Lz),
    E0=cfg.E0,
    seed=cfg.seed,
    poly_scale=cfg.poly_scale,
    fourier_scale=cfg.fourier_scale,
    n_fourier_terms=cfg.n_fourier_terms,
    use_polynomial=cfg.use_polynomial,
    use_fourier=cfg.use_fourier,
 )

world_model = create_harmonic_field_model(
    world_cfg,
    poly_coefficients=poly_coeffs,
    fourier_terms=fourier_terms,
 )

parent_field = sample_model_on_box(
    world_model,
    box_lengths=(cfg.Lx, cfg.Ly, cfg.Lz),
    grid_shape=(cfg.Nx, cfg.Ny, cfg.Nz),
    box_center=world_cfg.center,
 )

field_match_report = {
    "phi_max_abs_diff": float(np.max(np.abs(parent_field["phi"] - field["phi"]))),
    "E_max_abs_diff": float(np.max(np.abs(parent_field["E"] - field["E"]))),
}

phi_anchor = float(parent_field["phi"][0, 0, 0])
phi_recovered = recover_potential_from_field(
    parent_field,
    reference_index=(0, 0, 0),
    phi_reference=phi_anchor,
 )
phi_recovery_report = potential_recovery_metrics(parent_field["phi"], phi_recovered)

print("Boundary-potential workflow setup")
print("- CuPy available:", HAS_CUPY)
print("- Parent field agreement with existing notebook field:")
for key, value in field_match_report.items():
    print(f"  {key}: {value:.6e}")

print("\nPotential recovery from E with one reference potential:")
for key, value in phi_recovery_report.items():
    print(f"  {key}: {value:.6e}")

reference_boundary = extract_boundary_potential(parent_field["phi"])
print("\nBoundary arrays extracted for faces:")
for key, value in reference_boundary.items():
    print(f"  {key}: {value.shape}")


In [ ]:
# 정확한 경계 전위로 내부장 재현
single_box_grid = (41, 33, 35)
single_box_lengths = (cfg.Lx, cfg.Ly, cfg.Lz)

single_target = sample_model_on_box(
    world_model,
    box_lengths=single_box_lengths,
    grid_shape=single_box_grid,
    box_center=world_cfg.center,
)
single_boundary_array = pbr.boundary_dict_to_array(
    extract_boundary_potential(single_target["phi"]),
    single_target["phi"].shape,
)
single_solver = SpectralDirichletBoxSolver(single_box_grid, single_box_lengths)
single_phi = single_solver.solve(single_boundary_array)
single_reconstructed = {
    "x": single_target["x"],
    "y": single_target["y"],
    "z": single_target["z"],
    "phi": single_phi,
    "E": compute_electric_field_from_potential(
        single_phi,
        single_target["x"],
        single_target["y"],
        single_target["z"],
    ),
}
single_box_metrics = compute_reconstruction_metrics(
    single_target,
    single_reconstructed,
    compare_trim=1,
)

for key in [
    "phi_relative_l2",
    "phi_relative_l2_centered",
    "E_relative_l2",
    "E_vector_rmse",
    "mean_cosine_similarity",
]:
    print(f"{key}: {single_box_metrics[key]:.6e}")

mid_y = single_box_grid[1] // 2
target_mag = np.linalg.norm(np.stack(single_target["E"], axis=0), axis=0)
reconstructed_mag = np.linalg.norm(np.stack(single_reconstructed["E"], axis=0), axis=0)
error_mag = np.linalg.norm(
    np.stack(single_reconstructed["E"], axis=0) - np.stack(single_target["E"], axis=0),
    axis=0,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, values, title in zip(
    axes,
    [target_mag[:, mid_y, :], reconstructed_mag[:, mid_y, :], error_mag[:, mid_y, :]],
    ["Target |E|", "Reconstructed |E|", "Error |E|"],
):
    image = ax.imshow(values.T, origin="lower", aspect="auto")
    ax.set_title(title)
    fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# 부피와 위치 변화 분석
fixed_grid_shape = (31, 25, 27)
volume_fractions = [1.00, 0.75, 0.50, 0.35, 0.25, 0.15]

# These raw weights are renormalized inside compute_weighted_volume_scores so that their sum becomes 1.
# The weighted score is kept as a diagnostic sensitivity check, not as the final optimization rule.
raw_volume_weight = 0.50
raw_accuracy_weight = 0.50
step52_tau_E = 2.0e-4

volume_sweep = run_volume_translation_sweep(
    world_model,
    grid_shape=fixed_grid_shape,
    volume_fractions=volume_fractions,
    translations_per_axis=(3, 3, 3),
    max_centers=None,
    backend="auto",
    compare_trim=1,
 )

weighted_volume_result = compute_weighted_volume_scores(
    volume_sweep,
    volume_weight=raw_volume_weight,
    accuracy_weight=raw_accuracy_weight,
    error_key="E_relative_l2_mean",
 )

summary_rows = []
for row in weighted_volume_result["rows"]:
    summary_rows.append({
        "volume_fraction": row["volume_fraction"],
        "volume": row["volume"],
        "box_Lx": row["box_Lx"],
        "box_Ly": row["box_Ly"],
        "box_Lz": row["box_Lz"],
        "E_relative_l2_mean": row["E_relative_l2_mean"],
        "epsilon_E95_mean": row.get("epsilon_E95_mean", np.nan),
        "phi_relative_l2_mean": row["phi_relative_l2_mean"],
        "weighted_score": row["weighted_score"],
        "dimensions_changed": row["dimensions_changed"],
    })

print("Volume / translation sweep summary with weighted objective")
print(
    f"Raw weights -> volume: {raw_volume_weight:.3f}, accuracy: {raw_accuracy_weight:.3f}"
 )
print(
    f"Normalized weights -> volume: {weighted_volume_result['volume_weight']:.3f}, "
    f"accuracy: {weighted_volume_result['accuracy_weight']:.3f}"
 )
print(format_metric_table(
    summary_rows,
    [
        "volume_fraction",
        "volume",
        "box_Lx",
        "box_Ly",
        "box_Lz",
        "E_relative_l2_mean",
        "epsilon_E95_mean",
        "phi_relative_l2_mean",
        "weighted_score",
        "dimensions_changed",
    ],
))

plot_volume_sweep_summary(volume_sweep, metric="E_relative_l2_mean")
plot_volume_sweep_summary(volume_sweep, metric="phi_relative_l2_mean")
plot_volume_sweep_cloud(volume_sweep, metric="E_relative_l2")
plot_weighted_volume_score(weighted_volume_result)
plot_volume_box_lengths(weighted_volume_result)

best_mean_error_row = min(volume_sweep["summary"], key=lambda row: row["E_relative_l2_mean"])
best_weighted_row = weighted_volume_result["best_row"]
step52_feasible_rows = [
    row for row in weighted_volume_result["rows"]
    if row["E_relative_l2_mean"] <= step52_tau_E
]
step52_largest_feasible_row = (
    max(step52_feasible_rows, key=lambda row: row["volume_fraction"])
    if step52_feasible_rows
    else None
)

# From this point on, the notebook keeps using the largest feasible row as the experimental reference box.
# This avoids calling the smallest box "best" merely because it has the lowest discretization error.
best_volume_row = step52_largest_feasible_row if step52_largest_feasible_row is not None else best_weighted_row
best_volume_fraction = best_volume_row["volume_fraction"]
best_box_lengths = (
    best_volume_row["box_Lx"],
    best_volume_row["box_Ly"],
    best_volume_row["box_Lz"],
)

all_dimensions_change_for_smaller_boxes = all(
    row["dimensions_changed"]
    for row in weighted_volume_result["rows"]
    if row["volume_fraction"] < 1.0
 )
aspect_ratio_preserved_for_all = all(
    np.isclose(row["box_Lx"] / world_cfg.lengths[0], row["box_Ly"] / world_cfg.lengths[1])
    and np.isclose(row["box_Ly"] / world_cfg.lengths[1], row["box_Lz"] / world_cfg.lengths[2])
    for row in weighted_volume_result["rows"]
 )

print("\nBest pure-accuracy volume fraction (reference only)")
for key in ["volume_fraction", "volume", "E_relative_l2_mean", "phi_relative_l2_mean", "elapsed_sec"]:
    print(f"  {key}: {best_mean_error_row[key]:.6e}")

print("\nBest weighted volume fraction (diagnostic only)")
for key in ["volume_fraction", "volume", "box_Lx", "box_Ly", "box_Lz", "E_relative_l2_mean", "phi_relative_l2_mean", "weighted_score"]:
    print(f"  {key}: {best_weighted_row[key]:.6e}")

print(f"\nSTEP 5-2 constraint reinterpretation with tau_E = {step52_tau_E:.1e}")
if step52_largest_feasible_row is None:
    print("  No tested volume fraction satisfies the mean L2 error threshold; using weighted diagnostic row for later checks.")
else:
    for key in ["volume_fraction", "volume", "box_Lx", "box_Ly", "box_Lz", "E_relative_l2_mean", "epsilon_E95_mean"]:
        print(f"  {key}: {step52_largest_feasible_row[key]:.6e}")

print(f"\nAll smaller tested volumes changed Lx/Ly/Lz together: {all_dimensions_change_for_smaller_boxes}")
print(f"Aspect ratio preserved for all tested volumes: {aspect_ratio_preserved_for_all}")
print(f"Backend used by the sweep: {volume_sweep['backend']}")


In [ ]:
# 경계 잡음과 패치 근사 분석
noise_levels = [0.0, 0.001, 0.002, 0.005, 0.010, 0.020]
boundary_noise_result = run_boundary_noise_sweep(
    world_model,
    box_lengths=best_box_lengths,
    grid_shape=fixed_grid_shape,
    noise_levels=noise_levels,
    box_center=world_cfg.center,
    n_trials=4,
    backend="auto",
    compare_trim=1,
    seed=123,
 )

print("Boundary-voltage noise sweep")
print(format_metric_table(
    boundary_noise_result["summary"],
    [
        "noise_rms_fraction",
        "E_relative_l2_mean",
        "E_relative_l2_std",
        "phi_relative_l2_mean",
        "phi_relative_l2_std",
    ],
))

plot_noise_sweep(boundary_noise_result, metric="E_relative_l2_mean")
plot_noise_sweep(boundary_noise_result, metric="phi_relative_l2_mean")

patch_shapes = [(2, 2), (4, 4), (6, 6), (8, 8), (12, 12)]
boundary_patch_result = run_boundary_patch_sweep(
    world_model,
    box_lengths=best_box_lengths,
    grid_shape=fixed_grid_shape,
    patch_shapes=patch_shapes,
    box_center=world_cfg.center,
    backend="auto",
    compare_trim=1,
 )

print("\nPiecewise-constant boundary patch sweep")
print(format_metric_table(
    boundary_patch_result["records"],
    [
        "patch_y",
        "patch_z",
        "num_patch_values",
        "E_relative_l2",
        "phi_relative_l2",
        "mean_cosine_similarity",
    ],
))

plot_patch_sweep(boundary_patch_result, metric="E_relative_l2")
plot_patch_sweep(boundary_patch_result, metric="phi_relative_l2")


In [ ]:
# 경계 Fourier 모드 감쇠 분석
# =====================================================================
#  STEP 5-4 — Boundary Fourier-mode damping test
# =====================================================================
print("=" * 65)
print("STEP 5-4: Boundary Fourier-mode damping test")
print("=" * 65)

fourier_grid_shape = (29, 23, 25)
fourier_box_lengths = best_box_lengths
fourier_center = world_cfg.center

mode_field = sample_model_on_box(
    world_model,
    box_lengths=fourier_box_lengths,
    grid_shape=fourier_grid_shape,
    box_center=fourier_center,
)
solver = SpectralDirichletBoxSolver(fourier_grid_shape, fourier_box_lengths)

x_local = mode_field["x"] - mode_field["x"][0]
y_local = mode_field["y"] - mode_field["y"][0]
z_local = mode_field["z"] - mode_field["z"][0]
X2, Y2 = np.meshgrid(x_local, y_local, indexing="ij")
Lx_mode, Ly_mode, Lz_mode = fourier_box_lengths

fourier_modes = [(1, 1), (2, 2), (4, 3)]
fourier_decay_records = []

fig, ax = plt.subplots(figsize=(8, 5))
for mx, my in fourier_modes:
    boundary_unit = {
        "xmin": np.zeros((fourier_grid_shape[1], fourier_grid_shape[2])),
        "xmax": np.zeros((fourier_grid_shape[1], fourier_grid_shape[2])),
        "ymin": np.zeros((fourier_grid_shape[0], fourier_grid_shape[2])),
        "ymax": np.zeros((fourier_grid_shape[0], fourier_grid_shape[2])),
        "zmin": np.zeros((fourier_grid_shape[0], fourier_grid_shape[1])),
        "zmax": np.zeros((fourier_grid_shape[0], fourier_grid_shape[1])),
    }
    boundary_unit["zmin"] = np.sin(mx * np.pi * X2 / Lx_mode) * np.sin(my * np.pi * Y2 / Ly_mode)
    phi_mode = solver.solve(pbr.boundary_dict_to_array(boundary_unit, fourier_grid_shape))
    amp = np.sqrt(np.mean(phi_mode[1:-1, 1:-1, :] ** 2, axis=(0, 1)))
    amp_norm = amp / max(amp[0], 1e-15)
    k_eff = np.sqrt((mx * np.pi / Lx_mode) ** 2 + (my * np.pi / Ly_mode) ** 2)
    theory = np.sinh(k_eff * np.maximum(Lz_mode - z_local, 0.0)) / max(np.sinh(k_eff * Lz_mode), 1e-15)
    theory = theory / max(theory[0], 1e-15)

    fit_mask = (z_local > 0.05 * Lz_mode) & (z_local < 0.55 * Lz_mode) & (amp_norm > 1e-12)
    slope, intercept = np.polyfit(z_local[fit_mask], np.log(amp_norm[fit_mask]), 1)
    fourier_decay_records.append({
        "mode": f"({mx},{my})",
        "k_eff": k_eff,
        "fitted_decay_rate": -slope,
        "relative_rate_error": abs((-slope) - k_eff) / max(k_eff, 1e-15),
    })

    ax.plot(z_local, amp_norm, "o-", ms=3, label=f"수치 모드 ({mx},{my})")
    ax.plot(z_local, theory, "--", lw=1, label=f"이론 모드 ({mx},{my})")

ax.set_yscale("log")
ax.set_xlabel("경계 zmin으로부터의 거리 d (m)")
ax.set_ylabel("정규화된 전위 모드 진폭")
ax.set_title("경계 Fourier 모드의 내부 감쇠 검증")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Boundary-mode exponential damping summary")
print(format_metric_table(
    fourier_decay_records,
    ["mode", "k_eff", "fitted_decay_rate", "relative_rate_error"],
))


In [ ]:
# 전극 응답 행렬과 희소 전극 선택
# =====================================================================
#  STEP 5-5 — Electrode-response dictionary and sparse optimization
# =====================================================================
from generate_field.potential_boundary_reconstruction import (
    solve_laplace_dirichlet_many,
    solve_laplace_dirichlet_phi,
    apply_boundary_patch_approximation,
)

print("=" * 65)
print("STEP 5-5: Electrode-response matrix and sparse optimization")
print("=" * 65)

response_grid_shape = (17, 15, 15)
response_box_lengths = best_box_lengths
response_patch_shape = (4, 4)
response_trim_layers = 2
sparse_tau_E = 2.0e-3


def field_to_response_vector(E, mask):
    """Flatten E on a mask as [Ex, Ey, Ez] per evaluation point."""
    E_arr = np.stack(E, axis=0) if isinstance(E, (tuple, list)) else np.asarray(E)
    return E_arr[:, mask].T.reshape(-1)


def response_metrics_from_vector(y_true, y_pred, tau_E=sparse_tau_E):
    """Compute RMS-normalized local errors for a flattened response vector."""
    Et = y_true.reshape(-1, 3)
    Er = y_pred.reshape(-1, 3)
    diff = Er - Et
    E_rms = np.sqrt(np.mean(np.sum(Et * Et, axis=1))) + 1e-15
    local = np.sqrt(np.sum(diff * diff, axis=1)) / E_rms
    return {
        "E_relative_l2": float(np.linalg.norm(diff.ravel()) / (np.linalg.norm(Et.ravel()) + 1e-15)),
        "epsilon_E95": float(np.percentile(local, 95)),
        "epsilon_E99": float(np.percentile(local, 99)),
        "local_pass_fraction": float(np.mean(local <= tau_E)),
    }


def boundary_patch_edges(size, n_patch):
    """Return integer patch edges for one boundary-face axis."""
    return np.linspace(0, size, n_patch + 1, dtype=int)


def add_face_patch_column(shape, face_name, edge_a, edge_b, ia, ib):
    """Create one unit-voltage boundary column on a selected face patch."""
    nx, ny, nz = shape
    col = np.zeros(shape, dtype=float)
    sa = slice(edge_a[ia], edge_a[ia + 1])
    sb = slice(edge_b[ib], edge_b[ib + 1])
    if face_name == "xmin":
        col[0, sa, sb] = 1.0
    elif face_name == "xmax":
        col[-1, sa, sb] = 1.0
    elif face_name == "ymin":
        col[sa, 0, sb] = 1.0
    elif face_name == "ymax":
        col[sa, -1, sb] = 1.0
    elif face_name == "zmin":
        col[sa, sb, 0] = 1.0
    elif face_name == "zmax":
        col[sa, sb, -1] = 1.0
    else:
        raise ValueError(face_name)
    return col


def build_boundary_patch_columns(shape, patch_shape):
    """Build unit-voltage boundary columns for all six faces."""
    nx, ny, nz = shape
    pa, pb = patch_shape
    columns = []
    meta = []
    face_specs = [
        ("xmin", ny, nz), ("xmax", ny, nz),
        ("ymin", nx, nz), ("ymax", nx, nz),
        ("zmin", nx, ny), ("zmax", nx, ny),
    ]
    for face_name, na, nb in face_specs:
        ea = boundary_patch_edges(na, pa)
        eb = boundary_patch_edges(nb, pb)
        for ia in range(pa):
            for ib in range(pb):
                columns.append(add_face_patch_column(shape, face_name, ea, eb, ia, ib).reshape(-1))
                meta.append({"face": face_name, "ia": ia, "ib": ib})
    return np.column_stack(columns), meta


def compute_E_columns_from_phi_columns(phi_columns, shape, lengths):
    """Convert many potential columns into flattened electric-field response columns."""
    nx, ny, nz = shape
    x = np.linspace(0.0, lengths[0], nx)
    y = np.linspace(0.0, lengths[1], ny)
    z = np.linspace(0.0, lengths[2], nz)
    E_columns = []
    for j in range(phi_columns.shape[1]):
        phi_j = phi_columns[:, j].reshape(shape)
        E_j = compute_electric_field_from_potential(phi_j, x, y, z)
        E_columns.append(field_to_response_vector(E_j, response_eval_mask))
    return np.column_stack(E_columns)


def ridge_solve(A, y, ridge=1.0e-8):
    """Small-ridge least squares used for dense and restricted supports."""
    G = A.T @ A + ridge * np.eye(A.shape[1])
    b = A.T @ y
    return np.linalg.solve(G, b)


def omp_path(A, y, K_values, ridge=1.0e-8):
    """Orthogonal Matching Pursuit path with column normalization."""
    col_norm = np.linalg.norm(A, axis=0) + 1e-15
    An = A / col_norm
    max_K = int(max(K_values))
    support = []
    residual = y.copy()
    path = {}
    chosen = np.zeros(A.shape[1], dtype=bool)
    for K in range(1, max_K + 1):
        corr = An.T @ residual
        corr[chosen] = 0.0
        j = int(np.argmax(np.abs(corr)))
        support.append(j)
        chosen[j] = True
        c_support_n = ridge_solve(An[:, support], y, ridge=ridge)
        residual = y - An[:, support] @ c_support_n
        if K in K_values:
            c = np.zeros(A.shape[1], dtype=float)
            c[np.array(support)] = c_support_n / col_norm[np.array(support)]
            path[K] = c
    return path


def fista_lasso(A, y, lam, n_iter=400):
    """FISTA solver for 0.5||Ac-y||² + lam||c||₁."""
    L = np.linalg.norm(A, ord=2) ** 2 + 1e-15
    c = np.zeros(A.shape[1], dtype=float)
    z = c.copy()
    t = 1.0
    step = 1.0 / L
    for _ in range(n_iter):
        grad = A.T @ (A @ z - y)
        c_next = np.sign(z - step * grad) * np.maximum(np.abs(z - step * grad) - lam * step, 0.0)
        t_next = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
        z = c_next + ((t - 1.0) / t_next) * (c_next - c)
        c, t = c_next, t_next
    return c


def uniform_support_indices(M, K):
    """Choose approximately uniform electrode indices as a simple deterministic baseline."""
    return np.unique(np.linspace(0, M - 1, K, dtype=int))


def support_lstsq(A, y, support):
    c = np.zeros(A.shape[1], dtype=float)
    support = np.array(sorted(set(map(int, support))), dtype=int)
    if support.size > 0:
        c[support] = ridge_solve(A[:, support], y)
    return c

# Build target field and electrode dictionary on the response grid.
response_target_field = sample_model_on_box(
    world_model,
    box_lengths=response_box_lengths,
    grid_shape=response_grid_shape,
    box_center=world_cfg.center,
)
response_eval_mask = trim_mask_by_layers(
    np.ones(response_grid_shape, dtype=bool),
    response_trim_layers,
)
y_response = field_to_response_vector(response_target_field["E"], response_eval_mask)

boundary_columns, electrode_meta = build_boundary_patch_columns(response_grid_shape, response_patch_shape)
phi_response_columns = solve_laplace_dirichlet_many(boundary_columns, response_grid_shape, response_box_lengths)
A_response = compute_E_columns_from_phi_columns(phi_response_columns, response_grid_shape, response_box_lengths)

# Boundary-patch average baseline: patch the target boundary voltage and solve once.
phi_patch_avg = apply_boundary_patch_approximation(response_target_field["phi"], patch_shape=response_patch_shape)
phi_patch_avg_rec = solve_laplace_dirichlet_phi(phi_patch_avg, response_box_lengths)
E_patch_avg_rec = compute_electric_field_from_potential(
    phi_patch_avg_rec,
    response_target_field["x"], response_target_field["y"], response_target_field["z"],
)
y_patch_avg = field_to_response_vector(E_patch_avg_rec, response_eval_mask)

print(f"Response grid shape        : {response_grid_shape}")
print(f"Candidate patch shape      : {response_patch_shape}")
print(f"Number of candidate electrodes: {A_response.shape[1]}")
print(f"Number of evaluation points   : {np.sum(response_eval_mask)}")
print(f"Response matrix shape      : {A_response.shape}")

# Matrix diagnostics: SVD, effective rank, mutual coherence.
column_norm = np.linalg.norm(A_response, axis=0) + 1e-15
A_normalized = A_response / column_norm
singular_values = np.linalg.svd(A_normalized, compute_uv=False)
effective_rank = (np.sum(singular_values) ** 2) / (np.sum(singular_values ** 2) + 1e-15)
coherence_matrix = np.abs(A_normalized.T @ A_normalized)
np.fill_diagonal(coherence_matrix, 0.0)
mutual_coherence = float(np.max(coherence_matrix))

# Dense and sparse reconstructions.
dense_c = ridge_solve(A_response, y_response)
K_values = [4, 8, 12, 16, 24, 32, 48, 64]
K_values = [K for K in K_values if K <= A_response.shape[1]]
omp_coefficients = omp_path(A_response, y_response, K_values)

rng = np.random.default_rng(2026)
sparse_records = []
baseline_patch_metrics = response_metrics_from_vector(y_response, y_patch_avg, sparse_tau_E)
sparse_records.append({"method": "boundary_patch_average", "K": 6 * response_patch_shape[0] * response_patch_shape[1], "voltage_linf": np.nan, **baseline_patch_metrics})

dense_metrics = response_metrics_from_vector(y_response, A_response @ dense_c, sparse_tau_E)
sparse_records.append({"method": "dense_LS", "K": A_response.shape[1], "voltage_linf": float(np.max(np.abs(dense_c))), **dense_metrics})

for K in K_values:
    c_omp = omp_coefficients[K]
    m_omp = response_metrics_from_vector(y_response, A_response @ c_omp, sparse_tau_E)
    sparse_records.append({"method": "OMP", "K": K, "voltage_linf": float(np.max(np.abs(c_omp))), **m_omp})

    random_metrics = []
    random_vmax = []
    for _ in range(8):
        support = rng.choice(A_response.shape[1], size=K, replace=False)
        c_rand = support_lstsq(A_response, y_response, support)
        random_metrics.append(response_metrics_from_vector(y_response, A_response @ c_rand, sparse_tau_E))
        random_vmax.append(np.max(np.abs(c_rand)))
    sparse_records.append({
        "method": "random_support_mean",
        "K": K,
        "voltage_linf": float(np.mean(random_vmax)),
        "E_relative_l2": float(np.mean([m["E_relative_l2"] for m in random_metrics])),
        "epsilon_E95": float(np.mean([m["epsilon_E95"] for m in random_metrics])),
        "epsilon_E99": float(np.mean([m["epsilon_E99"] for m in random_metrics])),
        "local_pass_fraction": float(np.mean([m["local_pass_fraction"] for m in random_metrics])),
    })

    support_uniform = uniform_support_indices(A_response.shape[1], K)
    c_uniform = support_lstsq(A_response, y_response, support_uniform)
    m_uniform = response_metrics_from_vector(y_response, A_response @ c_uniform, sparse_tau_E)
    sparse_records.append({"method": "uniform_support", "K": K, "voltage_linf": float(np.max(np.abs(c_uniform))), **m_uniform})

# Lasso path. The columns are normalized for the optimization, then coefficients are rescaled.
lambda_max = float(np.max(np.abs(A_normalized.T @ y_response)))
lambda_values = lambda_max * np.logspace(-3.2, -0.6, 12)
lasso_records = []
for lam in lambda_values:
    c_lasso_n = fista_lasso(A_normalized, y_response, lam=lam, n_iter=350)
    c_lasso = c_lasso_n / column_norm
    support_size = int(np.sum(np.abs(c_lasso) > 1e-8))
    if support_size == 0:
        continue
    m_lasso = response_metrics_from_vector(y_response, A_response @ c_lasso, sparse_tau_E)
    lasso_records.append({"method": "Lasso", "lambda": lam, "K": support_size, "voltage_linf": float(np.max(np.abs(c_lasso))), **m_lasso})

# Print compact result tables.
print("\nResponse-matrix diagnostics")
print(format_metric_table([
    {
        "num_electrodes": A_response.shape[1],
        "effective_rank": effective_rank,
        "mutual_coherence": mutual_coherence,
        "sv_ratio_10_to_1": singular_values[min(9, len(singular_values)-1)] / singular_values[0],
    }
], ["num_electrodes", "effective_rank", "mutual_coherence", "sv_ratio_10_to_1"]))

print("\nSparse reconstruction summary")
summary_to_show = [r for r in sparse_records if r["method"] in ["boundary_patch_average", "dense_LS"]]
summary_to_show += [r for r in sparse_records if r["method"] == "OMP"]
print(format_metric_table(summary_to_show, ["method", "K", "E_relative_l2", "epsilon_E95", "local_pass_fraction", "voltage_linf"]))

print("\nLasso path summary")
print(format_metric_table(lasso_records, ["method", "lambda", "K", "E_relative_l2", "epsilon_E95", "local_pass_fraction", "voltage_linf"]))

# Plot 1. singular value spectrum.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(np.arange(1, len(singular_values) + 1), singular_values / singular_values[0], marker="o")
ax.set_title("전극 응답 행렬의 특이값 스펙트럼")
ax.set_xlabel("특이값 순서")
ax.set_ylabel(r"$\sigma_i / \sigma_1$")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 2. mutual coherence heatmap.
fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(coherence_matrix, origin="lower", aspect="auto")
ax.set_title("후보 전극 응답 열 사이의 mutual coherence")
ax.set_xlabel("전극 index")
ax.set_ylabel("전극 index")
plt.colorbar(im, ax=ax, label="정규화 내적 절댓값")
plt.tight_layout()
plt.show()

# Plot 3. K sweep.
fig, ax = plt.subplots(figsize=(8, 5))
for method in ["OMP", "random_support_mean", "uniform_support"]:
    rows = [r for r in sparse_records if r["method"] == method]
    ax.plot([r["K"] for r in rows], [r["epsilon_E95"] for r in rows], marker="o", label=method)
ax.axhline(sparse_tau_E, linestyle="--", linewidth=1.2, label=r"$\tau_E$")
ax.set_yscale("log")
ax.set_title("전극 수 K에 따른 95퍼센타일 국소 오차")
ax.set_xlabel("활성 전극 수 K")
ax.set_ylabel(r"$\epsilon_{E,95}$")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Plot 4. Pareto view between error and voltage scale.
fig, ax = plt.subplots(figsize=(8, 5))
for method in ["OMP", "random_support_mean", "uniform_support", "dense_LS"]:
    rows = [r for r in sparse_records if r["method"] == method]
    ax.scatter([r["voltage_linf"] for r in rows], [r["epsilon_E95"] for r in rows], label=method, s=55)
ax.set_yscale("log")
ax.set_title("오차와 최대 전압 크기의 Pareto 비교")
ax.set_xlabel(r"$\|c\|_\infty$ (상대 전압 단위)")
ax.set_ylabel(r"$\epsilon_{E,95}$")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Plot 5. selected electrode locations for a representative sparse design.
representative_K = 24 if 24 in omp_coefficients else K_values[len(K_values) // 2]
representative_c = omp_coefficients[representative_K]
selected_indices = np.flatnonzero(np.abs(representative_c) > 1e-12)
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True)
for ax, face_name in zip(axes.ravel(), ["xmin", "xmax", "ymin", "ymax", "zmin", "zmax"]):
    grid = np.zeros(response_patch_shape, dtype=float)
    for idx in selected_indices:
        meta = electrode_meta[idx]
        if meta["face"] == face_name:
            grid[meta["ia"], meta["ib"]] = representative_c[idx]
    im = ax.imshow(grid.T, origin="lower", aspect="auto")
    ax.set_title(f"{face_name} 면 선택 전극 전압")
    ax.set_xlabel("patch a")
    ax.set_ylabel("patch b")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(f"OMP K={representative_K}에서 선택된 전극 위치와 전압")
plt.tight_layout()
plt.show()

# Voltage quantization and one-pass local correction.
def quantize_coefficients(c, bits=None, step=None):
    c = np.asarray(c, dtype=float)
    vmax = np.max(np.abs(c)) + 1e-15
    if step is None:
        levels = 2 ** int(bits)
        step = 2.0 * vmax / (levels - 1)
    q = np.round(c / step) * step
    q[np.abs(c) < 1e-14] = 0.0
    return q, float(step)


def one_pass_quantized_correction(A, y, c_quantized, step, active_indices):
    c = c_quantized.copy()
    current_error = np.linalg.norm(A @ c - y)
    for idx in active_indices:
        best_value = c[idx]
        best_error = current_error
        for delta in [-step, step]:
            trial = c.copy()
            trial[idx] += delta
            err = np.linalg.norm(A @ trial - y)
            if err < best_error:
                best_error = err
                best_value = trial[idx]
        c[idx] = best_value
        current_error = best_error
    return c

quantization_records = []
active_indices = np.flatnonzero(np.abs(representative_c) > 1e-12)
for label, bits, step in [("8-bit", 8, None), ("10-bit", 10, None), ("12-bit", 12, None), ("고정 10 mV", None, 1.0e-2)]:
    cq, q_step = quantize_coefficients(representative_c, bits=bits, step=step)
    mq = response_metrics_from_vector(y_response, A_response @ cq, sparse_tau_E)
    cc = one_pass_quantized_correction(A_response, y_response, cq, q_step, active_indices)
    mc = response_metrics_from_vector(y_response, A_response @ cc, sparse_tau_E)
    quantization_records.append({"voltage_model": label, "step": q_step, "epsilon_E95_quantized": mq["epsilon_E95"], "epsilon_E95_corrected": mc["epsilon_E95"], "pass_fraction_corrected": mc["local_pass_fraction"]})

print("\nVoltage quantization with one-pass local correction")
print(format_metric_table(quantization_records, ["voltage_model", "step", "epsilon_E95_quantized", "epsilon_E95_corrected", "pass_fraction_corrected"]))

fig, ax = plt.subplots(figsize=(7.5, 4.8))
xpos = np.arange(len(quantization_records))
width = 0.36
ax.bar(xpos - width / 2, [r["epsilon_E95_quantized"] for r in quantization_records], width, label="양자화 직후")
ax.bar(xpos + width / 2, [r["epsilon_E95_corrected"] for r in quantization_records], width, label="1회 국소 보정 후")
ax.axhline(sparse_tau_E, linestyle="--", linewidth=1.2, label=r"$\tau_E$")
ax.set_xticks(xpos)
ax.set_xticklabels([r["voltage_model"] for r in quantization_records])
ax.set_yscale("log")
ax.set_title("전압 양자화와 국소 보정 후 국소 오차")
ax.set_ylabel(r"$\epsilon_{E,95}$")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 응용 목표장 6종 비교
from importlib import reload

import generate_field.potential_boundary_reconstruction as pbr

reload(pbr)

from generate_field.potential_boundary_reconstruction import (
    format_metric_table,
    plot_step6_case_comparison,
    plot_step6_error_slice,
    plot_step6_scale_error_summary,
    plot_step6_surface_center_ratio,
    run_step6_default_lab_sweeps,
)

step6_result = run_step6_default_lab_sweeps(backend="numpy")

step6_summary_rows = []
for row in step6_result["records"]:
    step6_summary_rows.append({
        "case_name": row["case_name"],
        "scale_multiplier": row["scale_multiplier"],
        "Lx": row["Lx"],
        "Ly": row["Ly"],
        "Lz": row["Lz"],
        "grid_nx": row["grid_nx"],
        "grid_ny": row["grid_ny"],
        "grid_nz": row["grid_nz"],
        "center_E_relative_l2": row["center_E_relative_l2"],
        "surface_E_relative_l2": row["surface_E_relative_l2"],
        "whole_E_relative_l2": row["whole_E_relative_l2"],
        "boundary_voltage_span": row["boundary_voltage_span"],
        "solve_seconds": row["solve_seconds"],
    })

print("STEP 6 fixed-cell scale sweep summary")
print(format_metric_table(
    step6_summary_rows,
    [
        "case_name",
        "scale_multiplier",
        "Lx",
        "Ly",
        "Lz",
        "grid_nx",
        "grid_ny",
        "grid_nz",
        "center_E_relative_l2",
        "surface_E_relative_l2",
        "whole_E_relative_l2",
        "boundary_voltage_span",
        "solve_seconds",
    ],
))

for sweep in step6_result["sweeps"]:
    plot_step6_scale_error_summary(sweep, metric="E_relative_l2")
    plot_step6_surface_center_ratio(sweep, metric="E_relative_l2")

plot_step6_case_comparison(step6_result, region="center", metric="E_relative_l2")
plot_step6_case_comparison(step6_result, region="surface", metric="E_relative_l2")

for sweep in step6_result["sweeps"]:
    largest_detail = sweep["details"][-1]
    plot_step6_error_slice(largest_detail, component="mag")


In [ ]:
# 강건 유효부피와 전압 양자화 검증
import json
import closure_analysis

closure_summary = closure_analysis.main()

print("Selected shape eta_tau:", closure_summary["selected_shape_original_grid"]["eta_tau"])
print("Selected shape epsilon_E95:", closure_summary["selected_shape_original_grid"]["epsilon_E95"])
print("High-resolution E relative-L2:", closure_summary["selected_shape_highres_grid"]["E_relative_l2"])
print(
    "12-bit epsilon_E95:",
    next(row["epsilon_E95"] for row in closure_summary["voltage_quantization"] if row["case"] == "12-bit"),
)


In [ ]:
# 핵심 결과 표 생성
import pandas as pd

core_results = pd.DataFrame(
    [
        {
            "analysis": "continuous boundary",
            "E_relative_l2": continuous_result["metrics"]["E_relative_l2"],
        },
        {
            "analysis": "10x10 patches",
            "E_relative_l2": discrete_result["metrics"]["E_relative_l2"],
        },
        {
            "analysis": "20x20 patches",
            "E_relative_l2": optimal_result["metrics"]["E_relative_l2"],
        },
        {
            "analysis": "exact Dirichlet boundary",
            "E_relative_l2": single_box_metrics["E_relative_l2"],
        },
    ]
)
display(core_results)
